# HydroKG results analysis

Loads the real CSV outputs from `hydrokg.cli.run_enhanced_training` (in `results/`) and produces the skill-trust and enhancement figures. No synthetic data anywhere in this notebook -- everything here reads outputs from an actual run against your CAMELS/LSTM data.

Run `python -m hydrokg.cli.run_enhanced_training ... --output_prefix results/<name>` first to produce the files this notebook reads.

In [ ]:
import pandas as pd

from hydrokg.evaluation import summarize_skill_trust, compute_deltas, enhancement_summary, stratified_violation_summary
from hydrokg.viz import (
    plot_skill_trust_scatter, plot_rule_violation_counts, plot_stratified_burden,
    plot_delta_kge_distribution, plot_skill_vs_consistency_gain,
)

RESULT_PREFIX = "../results/hydrokg_run1"  # match --output_prefix from your CLI run

## 1. Baseline audit: the skill-trust gap on your real basins

In [ ]:
baseline_results = pd.read_csv(f"{RESULT_PREFIX}_baseline_results.csv")
baseline_results[["basin_id", "kge", "violation_burden", "dominant_class"]].head()

In [ ]:
summarize_skill_trust(baseline_results)

In [ ]:
plot_skill_trust_scatter(baseline_results, save_path="../figures/skill_trust_scatter.png");

## 2. Stratified by aridity class (requires --stratification_db was passed)

In [ ]:
if "aridity_class" in baseline_results.columns:
    strat_summary = stratified_violation_summary(baseline_results, by="aridity_class")
    display(strat_summary)
    plot_stratified_burden(strat_summary, "aridity_class", save_path="../figures/burden_by_aridity.png");
else:
    print("No aridity_class column -- re-run the CLI with --stratification_db to get this breakdown.")

## 3. Enhanced vs. baseline: does graph-guided enhancement help?

In [ ]:
enhanced_results = pd.read_csv(f"{RESULT_PREFIX}_enhanced_results.csv")
summary = enhancement_summary(baseline_results, enhanced_results)
{k: v for k, v in summary.items() if k != "deltas"}

In [ ]:
deltas = compute_deltas(baseline_results, enhanced_results)
plot_delta_kge_distribution(deltas, save_path="../figures/delta_kge_distribution.png");
plot_skill_vs_consistency_gain(deltas, save_path="../figures/skill_vs_consistency_gain.png");

## Reading the joint plot

Upper-right quadrant (`delta_kge > 0` and `delta_violation_burden > 0`): basins where enhancement improved BOTH skill and physical consistency -- the result that would support the paper's central claim. Don't assume this quadrant dominates until you've actually looked -- report whatever the real distribution shows, including if most basins land elsewhere.